In [ ]:
import pandas as pd
import numpy as np


    # df.shape = (qatorlar_soni, ustunlar_soni)
    print(f"URL dan yuklandi: {df.shape[0]} ta qator, {df.shape[1]} ta ustun")

except Exception as e:
    # URL ishlamasa (internet yo'q, havola o'chgan) — xatolik sababi chiqadi
    print(f"URL ishlamadi ({e}). A usuldan foydalaning: CSV faylni qo'lda yuklang.")

    # Zaxira variant: faylni qo'lda yuklash
    from google.colab import files

    # Fayl tanlash oynasini ochamiz
    uploaded = files.upload()

    # Yuklangan fayllar dict ko'rinishida keladi — birinchi faylni olamiz
    filename = list(uploaded.keys())[0]

    # Faylni DataFrame'ga o'tkazamiz
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Yuklab olindi: {df.shape[0]} ta qator, {df.shape[1]} ta ustun")

In [ ]:

# MA'LUMOTLAR BILAN DASTLABKI TANISHISH — EDA (Exploratory Data Analysis)

# Dastlabki 5 qatorni ko'rsatadi — ma'lumotlar qanday ko'rinishda ekanini tezda tekshirish uchun
# df.tail() — oxirgi 5 qatorni, df.head(10) — dastlabki 10 qatorni ko'rsatadi
df.head()




# Jadvaldagi qatorlar va ustunlar sonini chiqaradi
# df.shape — (qatorlar_soni, ustunlar_soni) ko'rinishida tuple qaytaradi
print(f"O'lcham: {df.shape[0]} ta qator, {df.shape[1]} ta ustun")




# Har bir ustun haqida texnik ma'lumot:
#   - ustun nomi
#   - ma'lumot turi (int64, float64, object)
#   - bo'sh bo'lmagan qiymatlar soni (non-null count)
#   - xotira hajmi
# Bo'sh qiymatlar (null) borligini aniqlashda juda foydali
df.info()




# Faqat RAQAMLI ustunlar uchun statistik ko'rsatkichlar:
#   count  — qiymatlar soni
#   mean   — o'rtacha
#   std    — standart og'ish (qanchalik tarqalgan)
#   min    — eng kichik qiymat
#   25%    — Q1 (quyi kvartil)
#   50%    — Q2 (median)
#   75%    — Q3 (yuqori kvartil)
#   max    — eng katta qiymat
df.describe()

In [ ]:

# BO'SH QIYMATLARNI ANIQLASH (MISSING VALUES)


# Har bir ustundagi bo'sh qiymatlar sonini hisoblaymiz
missing = df.isnull().sum()

# Har bir ustundagi bo'sh qiymatlarning foiz ulushini hisoblaymiz
# formula: bo'sh_son / jami_qator * 100
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

# Ikki natijani birlashtirib, hisobot jadvali yasaymiz
missing_report = pd.DataFrame({
    'missing_count': missing,      # bo'sh qiymatlar soni
    'missing_pct': missing_pct     # foiz ulushi
})

# Faqat bo'sh qiymati BOR ustunlarni ko'rsatamiz
# (bo'sh_son > 0 bo'lganlarni filtrlash)
missing_report[missing_report['missing_count'] > 0]


# RAQAMLI USTUNLARDAGI BO'SH QIYMATLARNI TO'LDIRISH
# Median bilan to'ldiramiz — chunki median outlierga chidamli
# (o'rtacha esa katta outlierdan ta'sirlanib qoladi)


# Faqat raqamli (int, float) ustunlarni tanlaymiz
numerical_cols = df.select_dtypes(include=[np.number]).columns

for col in numerical_cols:
    # Faqat bo'sh qiymati bor ustunlarni to'ldiramiz
    if df[col].isnull().sum() > 0:

        # Ustunning median qiymatini hisoblaymiz
        median_val = df[col].median()

        # Bo'sh joylarni median bilan to'ldiramiz
        # inplace=True — o'zgarishni df'ga bevosita saqlaymiz (yangi o'zgaruvchi kerak emas)
        df[col].fillna(median_val, inplace=True)

        print(f"{col} ustuni median bilan to'ldirildi: {median_val}")



# MATNLI USTUNLARDAGI BO'SH QIYMATLARNI TO'LDIRISH
# Mode (eng ko'p takrorlanuvchi qiymat) bilan to'ldiramiz


# Faqat matnli (object/string) ustunlarni tanlaymiz
categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    # Faqat bo'sh qiymati bor ustunlarni to'ldiramiz
    if df[col].isnull().sum() > 0:

        # .mode() — eng ko'p takrorlanuvchi qiymatni qaytaradi
        # [0] — mode bir nechta bo'lishi mumkin, birinchisini olamiz
        mode_val = df[col].mode()[0]

        # Bo'sh joylarni mode bilan to'ldiramiz
        df[col].fillna(mode_val, inplace=True)

        print(f"{col} ustuni mode bilan to'ldirildi: {mode_val}")



# TEKSHIRISH — barcha bo'sh qiymatlar to'ldirilganmi?
# .sum().sum() — avval har ustun bo'yicha, keyin hammasi yig'iladi
# Natija 0 bo'lsa — tozalash muvaffaqiyatli

print(f"\nQolgan bo'sh qiymatlar soni: {df.isnull().sum().sum()}")

In [ ]:

# TAKRORIY QATORLARNI ANIQLASH VA O'CHIRISH


# Har bir qator boshqasi bilan to'liq bir xilmi — tekshiramiz
# .duplicated() — har qator uchun True/False qaytaradi (True = takroriy)
# .sum()        — True larni sanaydi (True = 1, False = 0)
n_duplicates = df.duplicated().sum()

print(f"Topilgan takroriy qatorlar: {n_duplicates}")



# Agar takroriy qatorlar bo'lsa — o'chiramiz

if n_duplicates > 0:
    # Takroriy qatorlarni o'chirib, tozalangan df ni qaytaramiz
    # .drop_duplicates() — birinchi uchragan qatorni qoldiradi, qolganlarini o'chiradi
    df = df.drop_duplicates()

    print(f"O'chirilgandan keyin: {df.shape[0]} ta qator qoldi")

else:
    # Takroriy qator topilmasa — xabar chiqaramiz
    print("Takroriy qatorlar yo'q — tozalash shart emas.")

In [ ]:

# SANA USTUNLARINI ANIQLASH VA TEKSHIRISH


# Barcha ustun nomlarini ko'rib chiqamiz
# Agar ustun nomida 'date' so'zi bo'lsa (katta-kichik harfdan qat'iy nazar) — chiqaramiz
print("O'zgartirishdan OLDIN:")
for col in df.columns:
    if 'date' in col.lower() or 'Date' in col:          # 'date' yoki 'Date' bor ustunlarni topamiz
        print(f"  {col}: {df[col].dtype}")               # hozirgi ma'lumot turi (object = matn)
        print(f"  Namuna qiymat: {df[col].iloc[0]}")     # birinchi qatordagi qiymatni ko'rsatamiz



# VAZIFA: Sana ustunlarini datetime formatiga o'tkazish


# pd.to_datetime() — matn ko'rinishidagi sanani datetime64 formatiga o'tkazadi
# datetime formatida bo'lsa — yil, oy, kun bo'yicha filtrlash va hisoblash mumkin bo'ladi

# Yuqorida chiqgan ustun nomlarini ko'rib, quyidagilarni to'g'rilang:
df['Order Date'] = pd.to_datetime(df['Order Date'])     # buyurtma sanasi
df['Ship Date']  = pd.to_datetime(df['Ship Date'])      # yetkazib berish sanasi



# VAZIFA: O'zgartirishni tekshirish


# O'zgartirishdan KEYIN ustun turlarini chiqaramiz
# Muvaffaqiyatli bo'lsa: dtype = datetime64[ns] ko'rinishi bo'lishi kerak
print("\nO'zgartirishdan KEYIN:")
for col in df.columns:
    if 'date' in col.lower() or 'Date' in col:
        print(f"  {col}: {df[col].dtype}")              # datetime64[ns] bo'lishi kerak
        print(f"  Namuna qiymat: {df[col].iloc[0]}")    # endi Timestamp formatida ko'rinadi

In [ ]:

# BARCHA USTUN NOMLARINI KO'RISH


# df dagi barcha ustun nomlarini chiqaramiz
# Keyingi vazifalarda to'g'ri nom ishlatish uchun avval bularni bilib olamiz
print("Barcha ustunlar:")
for col in df.columns:
    print(f"  {col}")



# VAZIFA 1: Har bir mijozning JAMI XARAJATI


# Mijozlarni 'Customer ID' bo'yicha guruhlаymiz
# Har bir mijoz uchun 'Sales' ustunini yig'amiz (.sum())
customer_spending = df.groupby('Customer ID')['Sales'].sum()

# Series nomini belgilaymiz — concat qilganda ustun nomi bo'ladi
customer_spending.name = 'total_spending'



# VAZIFA 2: Har bir mijozning BUYURTMALAR SONI (chastotasi)

# .size() — har bir guruhda nechta QATOR borligini sanaydi
# ya'ni har bir mijoz necha marta buyurtma berganini hisoblaydi
order_frequency = df.groupby('Customer ID').size()

# Series nomini belgilaymiz
order_frequency.name = 'order_frequency'



# VAZIFA 3: Har bir mijozning O'RTACHA BUYURTMA QIYMATI


# Har bir mijoz uchun 'Sales' ustunining o'rtachasini hisoblaymiz
# O'rtacha = jami xarajat / buyurtmalar soni
avg_order_value = df.groupby('Customer ID')['Sales'].mean()

# Series nomini belgilaymiz
avg_order_value.name = 'avg_order_value'



# VAZIFA 4: Uchala natijani BITTA JADVALGA BIRLASHTIRISH


# pd.concat() — bir xil indeksli (Customer ID) Series'larni
# yonma-yon (axis=1 = ustun bo'yicha) birlаshtiradi
customer_summary = pd.concat([
    customer_spending,    # jami xarajat
    order_frequency,      # buyurtmalar soni
    avg_order_value       # o'rtacha buyurtma qiymati
], axis=1)



# VAZIFA 5: NATIJANI KO'RISH VA STATISTIKA


# Dastlabki 10 mijozni ko'rsatamiz
print("Dastlabki 10 mijoz:")
print(customer_summary.head(10))

# Raqamli ustunlar bo'yicha statistik ko'rsatkichlar
# (min, max, mean, std va hokazo)
print("\nUmumiy statistika:")
customer_summary.describe()

In [ ]:

# TOZALANGAN MA'LUMOTLARNI CSV FAYL SIFATIDA SAQLASH


# Asosiy tozalangan DataFrame ni saqlaymiz
# index=False — qator raqamlari (0,1,2...) CSV ga yozilmaydi
#               (aks holda keraksiz ustun paydo bo'ladi)
df.to_csv('superstore_cleaned.csv', index=False)
print("✓ superstore_cleaned.csv saqlandi!")


# Mijozlar xulosasi DataFrame ni saqlaymiz
# Bu fayl keyingi darsda ML modellar uchun ishlatiladi
# index=True (standart) — 'Customer ID' indeks sifatida yoziladi
customer_summary.to_csv('customer_summary.csv')
print("✓ customer_summary.csv saqlandi!")